In [ ]:
import cv2
import numpy as np

In [2]:
import torch
import torch.nn as nn
from torch.optim import Adam, SGD

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
import os

In [1]:
# os.listdir('sample_data')

In [ ]:
detector = MTCNN()

In [ ]:
for img in os.listdir("img_folder"):
    arr = cv2.imread("img_folder/"+ img)
    results = detector.detect_faces(arr)
    if results:
        x, y, width, height = results[0]['box']
        x1, y1 = max(0, x), max(0, y)
        x2, y2 = x1 + width, y1 + height
        face_crop = img[y1:y2, x1:x2]
        cv2.imwrite('newfolder/'+img, face_crop)

In [ ]:
ls = []
for img in os.listdir("newfolder"):
    arr = cv2.imread("newfolder/"+ img)
    arr = cv2.resize(arr, (224, 224))
    ls.append(arr)

In [ ]:
arX = np.stack(ls, axis=0)

In [ ]:
X = torch.FloatTensor(arX)

In [ ]:
catg = ['tom','tom','tom','tom','tom','brad','brad','brad','brad','brad']

In [3]:
Y = torch.LongTensor([0,0,0,0,0,1,1,1,1,1])

In [ ]:
from torchvision import models, transforms, datasets

In [ ]:
mod = models.vgg16(pretrained=True)

In [ ]:
mod.classifier[6] = nn.Linear(4096, 2)

In [ ]:
for param in mod.parameters():
    param.requires_grad = False

In [ ]:
for p in mod.classifier[6].parameters():
  p.requires_grad = True

In [ ]:
opt = Adam(mod.parameters(), lr=0.001)
lossfn = nn.CrossEntropyLoss()

In [ ]:
for epoch in range(3):
    opt.zero_grad()
    pred = mod(X)
    loss = lossfn(pred,Y)
    loss.backward()
    opt.step()
    print(loss.item())

In [ ]:
torch.save(mod.state_dict(), 'model.pth')